# 🧩 Mini-Lab: From Smelly Notebook to a Pure Function

**Module 2: The Python That Runs Your Pipeline** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Identify** notebook smells that break reuse: cell-order state, globals, magic constants
2. **Extract** the classify step into `classify_message(text) -> dict`
3. **Write** a docstring and remove hidden globals so the function can be imported

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Notebook vs reusable code | Hidden cell-order state, globals, magic constants break reuse |
| Pure function | Inputs via parameters, results via return; printing is the caller's job |


---
## First: why this notebook breaks on Monday


## What Is a "Notebook Smell"?

A **code smell** is a surface-level pattern that hints at a deeper problem. In notebooks, three smells are especially dangerous because they make code **unreproducible** and **non-reusable**:

| Smell | What It Looks Like | Why It Breaks |
|-------|-------------------|---------------|
| **Cell-order dependence** | Result changes if you run cells out of order | Teammates can't reproduce your results |
| **Hidden globals** | A variable set in one cell is silently used 20 cells later | Restart kernel → `NameError` |
| **Magic constants** | Hard-coded numbers/strings with no name or explanation | Nobody knows what `0.85` means next month |

Let's see each one in action.

## Smell 1: Cell-Order Dependence

Run the two cells below **in order** — everything works. Then try running Cell B *before* Cell A after a kernel restart.

In [1]:
# --- Cell A: set up data ---
documents = [
    "LLMs generate text token by token.",
    "RAG retrieves context before generating.",
    "Embeddings capture semantic meaning.",
]
print(f"Loaded {len(documents)} documents")

Loaded 3 documents


In [2]:
# --- Cell B: process data (depends on Cell A) ---
avg_length = sum(len(d) for d in documents) / len(documents)
print(f"Average document length: {avg_length:.1f} characters")

Average document length: 36.7 characters


☝️ Cell B only works if Cell A has already run **in this kernel session**. If you restart the kernel and jump straight to Cell B, you get a `NameError` because `documents` doesn't exist yet.

**The deeper problem:** In a long notebook with 50+ cells, it's easy to accidentally run cells out of order during exploration. The notebook "works on my machine" but fails for anyone else — or even for you tomorrow.

## Smell 2: Hidden Globals

A variable is set once, then quietly reused much later — with nothing in between making the dependency obvious.

In [3]:
# Imagine this cell was run many cells ago, during exploration...
model_name = "gpt-4o-mini"
max_tokens = 500
threshold = 0.85

In [4]:
# ...then 30 cells later, this cell silently depends on all three globals
def build_request(prompt):
    return {
        "model": model_name,         # where did this come from?
        "max_tokens": max_tokens,     # set where?
        "messages": [{"role": "user", "content": prompt}],
    }

request = build_request("Explain RAG in one sentence.")
print(request)

{'model': 'gpt-4o-mini', 'max_tokens': 500, 'messages': [{'role': 'user', 'content': 'Explain RAG in one sentence.'}]}


☝️ The function *looks* like it only takes `prompt`, but it secretly depends on `model_name` and `max_tokens` defined far away. If someone imports this function into another file, those globals won't exist.

**The fix preview:** Make the function take everything it needs as parameters. We'll do exactly this in the next mini-lab.

## Smell 3: Magic Constants

Hard-coded numbers or strings with no explanation — you wrote them during a quick experiment and never named them.

In [5]:
# Smelly version — what do these numbers mean?
scores = [0.92, 0.67, 0.88, 0.45, 0.73, 0.91]

results = []
for s in scores:
    if s > 0.85:          # magic number — why 0.85?
        results.append(s)

print(f"Passing scores: {results}")
print(f"Kept {len(results)} out of {len(scores)}")

Passing scores: [0.92, 0.88, 0.91]
Kept 3 out of 6


☝️ What does `0.85` represent? A similarity threshold? A confidence cutoff? A grade boundary? Nobody knows — including future-you.

Compare with the named version:

In [6]:
# Clean version — the constant has a name
SIMILARITY_THRESHOLD = 0.85

scores = [0.92, 0.67, 0.88, 0.45, 0.73, 0.91]

results = [s for s in scores if s > SIMILARITY_THRESHOLD]

print(f"Scores above {SIMILARITY_THRESHOLD}: {results}")
print(f"Kept {len(results)} out of {len(scores)}")

Scores above 0.85: [0.92, 0.88, 0.91]
Kept 3 out of 6


Now the intent is clear, and if you need to change the threshold, there's exactly one place to do it.

## Audit Exercise: Spot the Smells

Read the notebook-style code below. It has **all three smells** baked in. The audit cell after it identifies each one.

In [7]:
# ---- Simulated "messy notebook" in a single cell ----
# (Imagine each section was a separate cell in a real notebook)

# "Cell 1" — run during morning exploration
data = ["AI is transforming industries", "Machine learning needs data", "LLMs use transformers"]
lang = "en"

# "Cell 2" — a quick experiment
cleaned = [d.lower().strip() for d in data]

# "Cell 3" — much later, uses globals from cells 1 and 2
def analyze(text_list):
    output = []
    for t in text_list:
        words = t.split()
        if len(words) > 3:             # magic constant: why 3?
            output.append({
                "text": t,
                "word_count": len(words),
                "language": lang,       # hidden global from "Cell 1"
            })
    return output

results = analyze(cleaned)             # depends on "Cell 2" global
for r in results:
    print(r)

{'text': 'ai is transforming industries', 'word_count': 4, 'language': 'en'}
{'text': 'machine learning needs data', 'word_count': 4, 'language': 'en'}


In [8]:
# ---- Audit report ----
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

audit = """
### 🔍 Audit Results

| # | Line / Area | Smell | Why It's a Problem |
|---|-------------|-------|--------------------|
| 1 | `results = analyze(cleaned)` | **Cell-order dependence** | `cleaned` only exists if "Cell 2" ran before "Cell 3" |
| 2 | `"language": lang` inside `analyze()` | **Hidden global** | `lang` is set in "Cell 1" but used inside a function — importing this function elsewhere fails |
| 3 | `if len(words) > 3` | **Magic constant** | Why 3? Is it a minimum sentence length? A filter threshold? Unnamed and unexplained |

**Score: 3 smells found** — this notebook would break on a kernel restart or if moved to a `.py` file.
"""

md(audit)


### 🔍 Audit Results

| # | Line / Area | Smell | Why It's a Problem |
|---|-------------|-------|--------------------|
| 1 | `results = analyze(cleaned)` | **Cell-order dependence** | `cleaned` only exists if "Cell 2" ran before "Cell 3" |
| 2 | `"language": lang` inside `analyze()` | **Hidden global** | `lang` is set in "Cell 1" but used inside a function — importing this function elsewhere fails |
| 3 | `if len(words) > 3` | **Magic constant** | Why 3? Is it a minimum sentence length? A filter threshold? Unnamed and unexplained |

**Score: 3 smells found** — this notebook would break on a kernel restart or if moved to a `.py` file.


## What Is a Pure Function?

A **pure function** follows two rules:

1. **Same inputs → same output.** It reads nothing except its parameters.
2. **No side effects.** It doesn't modify globals, write files, or change things outside itself.

Why does this matter for AI engineering?

- You can **import** a pure function into a script, a FastAPI endpoint, or a test — it just works.
- You can **test** it by calling it with known inputs and checking the output.
- You can **reuse** it across notebooks without worrying about hidden state.

## The Problem: Notebook-Style Code

Recall from the previous mini-lab that notebook code often relies on globals. Here's a typical example — inline logic that scores documents by keyword overlap:

In [1]:
# --- Notebook-style: globals everywhere ---
query = "How do transformers handle long sequences?"
documents = [
    "Transformers use attention to process sequences in parallel.",
    "RNNs handle sequences one token at a time.",
    "Positional encoding tells transformers about word order.",
]
threshold = 0.3

# Inline scoring logic
query_words = set(query.lower().split())
scored = []
for doc in documents:
    doc_words = set(doc.lower().split())
    overlap = len(query_words & doc_words) / len(query_words)
    if overlap >= threshold:
        scored.append({"text": doc, "score": round(overlap, 2)})

scored.sort(key=lambda x: x["score"], reverse=True)
print("Matching documents:")
for item in scored:
    print(f"  [{item['score']}] {item['text']}")

Matching documents:


This works — but it has the smells we identified last time:

- `query`, `documents`, and `threshold` are **globals** — the logic can't run without them.
- The scoring logic is **inline** — you can't call it from another file or test it in isolation.
- If you change `threshold` three notebooks later, this cell silently changes behavior.

Let's fix this by extracting a pure function.

## Step 1: Identify Inputs and Output

Before writing the function, ask:

- **What does this logic need?** → a query, a list of documents, and a threshold
- **What does it produce?** → a sorted list of matching documents with scores

Every piece of data the logic touches becomes a **parameter**. The result becomes the **return value**.

## Step 2: Extract the Pure Function

We move the logic into a function where *everything* it needs comes through parameters:

In [2]:
def score_documents(query, documents, threshold=0.3):
    """Score documents by keyword overlap with the query.

    Returns a sorted list of dicts with 'text' and 'score' keys,
    including only documents at or above the threshold.
    """
    query_words = set(query.lower().split())
    results = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(query_words & doc_words) / len(query_words)
        if overlap >= threshold:
            results.append({"text": doc, "score": round(overlap, 2)})
    results.sort(key=lambda x: x["score"], reverse=True)
    return results

Notice what changed:

| Before (notebook-style) | After (pure function) |
|------------------------|----------------------|
| `query` is a global | `query` is a parameter |
| `documents` is a global | `documents` is a parameter |
| `threshold` is a global | `threshold` is a parameter with a default |
| Results printed inline | Results **returned** |
| Can't reuse or test | Can import, call, and test anywhere |

## Step 3: Call It Like Any Other Function

Now the caller decides what data to pass — no globals needed:

In [3]:
documents = [
    "Transformers use attention to process sequences in parallel.",
    "RNNs handle sequences one token at a time.",
    "Positional encoding tells transformers about word order.",
]

# Call with one query
results = score_documents("How do transformers handle long sequences?", documents)
print("Matches:")
for r in results:
    print(f"  [{r['score']}] {r['text']}")

# Call again with a stricter threshold — same function, different config
strict = score_documents("How do transformers handle long sequences?", documents, threshold=0.4)
print(f"\nWith threshold=0.4: {len(strict)} match(es)")

Matches:

With threshold=0.4: 0 match(es)


The function is **pure**: same inputs always produce the same output, and it touches nothing outside itself.

## The Purity Test

A quick way to check if your function is pure — it should pass all three:

| Test | Question | `score_documents` |
|------|----------|-------------------|
| **No globals** | Does it read any variable not in its parameter list? | ✅ No |
| **No side effects** | Does it modify anything outside itself (globals, files, prints)? | ✅ No |
| **Deterministic** | Same arguments → same return value, every time? | ✅ Yes |

## Impure vs. Pure — Side by Side

Let's see one more quick comparison to make the pattern stick. Here's a common notebook pattern that *looks* like a function but is actually impure:

In [4]:
# --- IMPURE: secretly reads a global ---
MODEL = "gpt-4o-mini"

def build_request_impure(prompt):
    return {"model": MODEL, "messages": [{"role": "user", "content": prompt}]}


# --- PURE: everything comes through parameters ---
def build_request_pure(prompt, model):
    return {"model": model, "messages": [{"role": "user", "content": prompt}]}


# The pure version works without any globals
print("Impure:", build_request_impure("Hello"))
print("Pure:  ", build_request_pure("Hello", "gpt-4o-mini"))

Impure: {'model': 'gpt-4o-mini', 'messages': [{'role': 'user', 'content': 'Hello'}]}
Pure:   {'model': 'gpt-4o-mini', 'messages': [{'role': 'user', 'content': 'Hello'}]}


Both produce the same dict *right now* — but only the pure version survives being imported into a different file where `MODEL` doesn't exist.

## 🎯 Summary

### Key Takeaways

1. **Pure function** — a function whose output depends only on its arguments and that has no side effects; it's the smallest unit of reusable code
2. **Extract by asking two questions** — "What does this logic need?" (→ parameters) and "What does it produce?" (→ return value)
3. **Globals become parameters** — every external variable the logic touches should be passed in explicitly, not read from the notebook's global scope
4. **Return, don't print** — a pure function returns its result so the caller decides what to do with it (print, store, test, pipe to an API)